# 02 Counterfactual Explanations
Step-by-step generation of each counterfactual figure.


## Setup


In [ ]:
from pathlib import Path
import sys

ROOT = Path('..').resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from notebook_runner import run_notebook_script


## Configuration


In [ ]:
SCRIPT_NAME = '02_counterfactual'

# Recommended defaults
# - temporal_clusters works with dataset_kwargs={}
# - for netzschleuder records (e.g. sp_high_school), pass dataset_kwargs={'network': 'proximity'}
dataset_name = 'temporal_clusters'
dataset_kwargs = {}
target_attr_override = None

overrides = {
    'dataset_name': dataset_name,
    'dataset_kwargs': dataset_kwargs,
    'target_attr_override': target_attr_override,
    'render_all_plots': False,  # plot figure-by-figure below
    # 'target_node_idx': 2,
    # 'max_k': 300,
}


## Compute Shared State
This runs data loading, training/loading, and counterfactual ranking once.


In [ ]:
state = run_notebook_script(SCRIPT_NAME, overrides=overrides)

plot_dir = Path(state['plot_dir'])
target_node_idx = int(state['target_node_idx'])
result = state['result']

print('Target node:', target_node_idx)
print('Flip success:', bool(result.success), 'removed:', int(result.n_removed))
print('Plot dir:', plot_dir)

palette = list(state.get('class_colors', state['DEFAULT_CLASS_COLORS']))
print('Palette:', {'eventblue': palette[0], 'snapshotorange': palette[1], 'edgegray': palette[2]})

### Plot 1: Class Probabilities Before/After Deletions
Shows how the predicted distribution changes after deleting the selected HO edges.


In [ ]:
state['plot_probs_bar'](
    p_before=state['p0'],
    p_after=state['p1'],
    title=f'Node {target_node_idx}: probs before/after (k={int(result.n_removed)})',
    backend='plotly',
    save_path=plot_dir / f'target_{target_node_idx}_class_probs.pdf',
)


### Plot 2: Probability Curve (Counterfactual Order, Remove)
Delete top-ranked HO edges cumulatively in counterfactual order.


In [ ]:
if state['k_vals'] is not None:
    state['plot_prob_curve'](
        k_values=state['k_vals'],
        probs=state['probs'],
        vline_k=int(result.n_removed) if bool(result.success) else None,
        title=f'Node {target_node_idx}: probability curve',
        backend='plotly',
        class_colors=state.get('class_colors', state['DEFAULT_CLASS_COLORS']),
        save_path=plot_dir / f'target_{target_node_idx}_prob_curve.pdf',
    )
else:
    print('No counterfactual remove-curve available.')


### Plot 3: Keep-Only Curve (Counterfactual Order)
Keep only the top-k ranked HO edges; remove the rest.


In [ ]:
if state['k_keep'] is not None:
    state['plot_prob_curve'](
        k_values=state['k_keep'],
        probs=state['probs_keep'],
        title=f'Node {target_node_idx}: prob vs kept edges (cf order)',
        x_label='Number of higher-order edges kept (k)',
        backend='plotly',
        class_colors=state.get('class_colors', state['DEFAULT_CLASS_COLORS']),
        save_path=plot_dir / f'target_{target_node_idx}_prob_curve_keep_only_cf.pdf',
    )
else:
    print('No counterfactual keep-only curve available.')


### Plot 4: Probability Curve (Margin Order, Remove)
Alternative ranking based on greedy margin drop.


In [ ]:
if state['k_vals_margin'] is not None:
    state['plot_prob_curve'](
        k_values=state['k_vals_margin'],
        probs=state['probs_margin'],
        title=f'Node {target_node_idx}: prob curve (margin-based order)',
        backend='plotly',
        class_colors=state.get('class_colors', state['DEFAULT_CLASS_COLORS']),
        save_path=plot_dir / f'target_{target_node_idx}_prob_curve_margin_order.pdf',
    )
else:
    print('Margin-based order was empty.')


### Plot 5: Keep-Only Curve (Margin Order)


In [ ]:
if state['k_keep_margin'] is not None:
    state['plot_prob_curve'](
        k_values=state['k_keep_margin'],
        probs=state['probs_keep_margin'],
        title=f'Node {target_node_idx}: prob vs kept edges (margin order)',
        x_label='Number of higher-order edges kept (k)',
        backend='plotly',
        class_colors=state.get('class_colors', state['DEFAULT_CLASS_COLORS']),
        save_path=plot_dir / f'target_{target_node_idx}_prob_curve_keep_only_margin_order.pdf',
    )
else:
    print('Margin keep-only curve unavailable.')


### Plot 6: Probability Curve (Data-Only Order, Remove)
Ranking based only on data-derived HO evidence.


In [ ]:
if state['k_vals_data_only'] is not None:
    state['plot_prob_curve'](
        k_values=state['k_vals_data_only'],
        probs=state['probs_data_only'],
        title=f'Node {target_node_idx}: prob curve (data-only order)',
        backend='plotly',
        class_colors=state.get('class_colors', state['DEFAULT_CLASS_COLORS']),
        save_path=plot_dir / f'target_{target_node_idx}_prob_curve_data_only_order.pdf',
    )
else:
    print('Data-only order was empty.')


### Plot 7: Keep-Only Curve (Data-Only Order)


In [ ]:
if state['k_keep_data_only'] is not None:
    state['plot_prob_curve'](
        k_values=state['k_keep_data_only'],
        probs=state['probs_keep_data_only'],
        title=f'Node {target_node_idx}: prob vs kept edges (data-only order)',
        x_label='Number of higher-order edges kept (k)',
        backend='plotly',
        class_colors=state.get('class_colors', state['DEFAULT_CLASS_COLORS']),
        save_path=plot_dir / f'target_{target_node_idx}_prob_curve_keep_only_data_only_order.pdf',
    )
else:
    print('Data-only keep-only curve unavailable.')


### Plot 8: Overlay (Remove Curves)
Compare counterfactual vs margin vs data-only ranking orders.


In [ ]:
curves = []
if state['k_vals_pad'] is not None:
    curves.append(dict(label='cf', k_values=state['k_vals_pad'], probs=state['probs_pad'], line=dict(dash='solid')))
if state['k_vals_margin_pad'] is not None:
    curves.append(dict(label='margin', k_values=state['k_vals_margin_pad'], probs=state['probs_margin_pad'], line=dict(dash='dot')))
if state['k_vals_data_only_pad'] is not None:
    curves.append(dict(label='data_only', k_values=state['k_vals_data_only_pad'], probs=state['probs_data_only_pad'], line=dict(dash='dashdot')))

if curves:
    state['plot_prob_curves_overlay'](
        curves=curves,
        title=f'Node {target_node_idx}: cf vs margin/data-only order',
        x_label='Number of higher-order edges removed (k)',
        backend='plotly',
        class_colors=state.get('class_colors', state['DEFAULT_CLASS_COLORS']),
        max_k=state['max_k_common'],
        save_path=plot_dir / f'target_{target_node_idx}_prob_curve_cf_vs_margin_data_only.pdf',
    )
else:
    print('No overlay remove-curves available.')


### Plot 9: Overlay (Keep-Only Curves)


In [ ]:
curves_keep = []
if state['k_keep_pad'] is not None:
    curves_keep.append(dict(label='cf', k_values=state['k_keep_pad'], probs=state['probs_keep_pad'], line=dict(dash='solid')))
if state['k_keep_margin_pad'] is not None:
    curves_keep.append(dict(label='margin', k_values=state['k_keep_margin_pad'], probs=state['probs_keep_margin_pad'], line=dict(dash='dot')))
if state['k_keep_data_only_pad'] is not None:
    curves_keep.append(dict(label='data_only', k_values=state['k_keep_data_only_pad'], probs=state['probs_keep_data_only_pad'], line=dict(dash='dashdot')))

if curves_keep:
    state['plot_prob_curves_overlay'](
        curves=curves_keep,
        title=f'Node {target_node_idx}: keep-only cf vs margin/data-only order',
        x_label='Number of higher-order edges kept (k)',
        backend='plotly',
        class_colors=state.get('class_colors', state['DEFAULT_CLASS_COLORS']),
        max_k=state['max_k_keep_common'],
        save_path=plot_dir / f'target_{target_node_idx}_prob_curve_keep_only_cf_vs_margin_data_only.pdf',
    )
else:
    print('No overlay keep-only curves available.')


### Plot 10: Removed-Edge Subgraph
Tiny graph containing only deleted HO edges.


In [ ]:
if result.removed_edges_as_node_ids:
    state['plot_removed_edges_graph'](
        removed_edges_as_node_ids=result.removed_edges_as_node_ids,
        target_node_idx=target_node_idx,
        save_path=plot_dir / f'target_{target_node_idx}_removed_edges.pdf',
    )
else:
    print('No removed edges to visualize.')
